# PerceptSent Agreement — Ground Truth & Evaluation (Cross-Validation)

**Purpose:**
1. Characterise the 12 agreement-filtered subsets (sigma × population threshold).
2. Produce reproducible **5-Fold Cross-Validation splits** (`random_state=42`) for each CSV.
3. Evaluate MLLM annotation outputs against the ground truth across folds.
4. Save all metrics, confusion matrices, and figures to disk.

In [ ]:
## 1  Environment
import sys
from pathlib import Path

try:
    ROOT = Path(__file__).resolve().parent.parent
except NameError:
    _cwd = Path.cwd()
    ROOT = _cwd.parent if _cwd.name == "notebooks" else _cwd

sys.path.insert(0, str(ROOT / "src"))

import warnings
warnings.filterwarnings("ignore")

import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    classification_report, confusion_matrix,
    f1_score, cohen_kappa_score, roc_auc_score,
    accuracy_score, mean_absolute_error,
)

sns.set_theme(style="whitegrid", context="paper", font_scale=1.2)
PALETTE = sns.color_palette("muted")

AGREEMENT_DIR = ROOT / "data" / "perceptsent-agreement"
OUTPUT_DIR    = ROOT / "outputs"
FIGURES_DIR   = ROOT / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

def _save_fig(fig, stem: str) -> None:
    """Save figure as both PDF and PNG at 300 dpi."""
    for ext in ("pdf", "png"):
        fig.savefig(FIGURES_DIR / f"{stem}.{ext}", bbox_inches="tight", dpi=300)
    print(f"Saved → {stem}  [.pdf + .png]")

SENTIMENT_MAPS: dict[str, dict[str, int]] = {
    "p2neg": {
        "Positive":         1,
        "SlightlyPositive": 1,
        "Neutral":          0,
        "SlightlyNegative": 0,
        "Negative":         0,
    },
    "p2plus": {
        "Positive":         0,
        "SlightlyPositive": 0,
        "Neutral":          0,
        "SlightlyNegative": 1,
        "Negative":         1,
    },
    "p3": {
        "Positive":         2,
        "SlightlyPositive": 2,
        "Neutral":          0,
        "SlightlyNegative": 1,
        "Negative":         1,
    },
    "p5": {
        "Positive":         4,
        "SlightlyPositive": 3,
        "Neutral":          2,
        "SlightlyNegative": 1,
        "Negative":         0,
    },
}

PROBLEM_TYPE: dict[str, str] = {
    "p2neg":  "binary",
    "p2plus": "binary",
    "p3":     "multiclass-3",
    "p5":     "ordinal-5",
}

p_order = ["p2neg", "p2plus", "p3", "p5"]

print(f"Root             : {ROOT}")
print(f"Agreement CSVs   : {AGREEMENT_DIR}")
print(f"Figures          : {FIGURES_DIR}")

In [ ]:
## 2  Load & Characterise All CSVs
_PAT = re.compile(r"percept_dataset_sigma(\d+)_(p\w+)\.csv")

csv_files = sorted(AGREEMENT_DIR.glob("*.csv"))
print(f"Found {len(csv_files)} CSV files:\n")

datasets: dict[str, pd.DataFrame] = {}
meta_rows = []

for fpath in csv_files:
    m = _PAT.match(fpath.name)
    if not m:
        continue
    sigma, p_type = int(m.group(1)), m.group(2)
    key = f"sigma{sigma}_{p_type}"

    df = pd.read_csv(fpath)
    n_total      = len(df)
    label_counts = df["ground_truth"].value_counts().sort_index().to_dict()
    n_classes    = df["ground_truth"].nunique()

    datasets[key] = df
    meta_rows.append({
        "key":       key,
        "sigma":     sigma,
        "p_type":    p_type,
        "problem":   PROBLEM_TYPE.get(p_type, "unknown"),
        "filename":  fpath.name,
        "n_total":   n_total,
        "n_classes": n_classes,
        "label_dist": label_counts,
    })
    dist_str = "  ".join(f"label {k}={v}" for k, v in label_counts.items())
    print(f"  {key:28s}  n={n_total:5,}  classes={n_classes}  [{dist_str}]")

meta_df = pd.DataFrame(meta_rows)

# ── Dataset sizes ─────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

pivot_size = meta_df.pivot(index="sigma", columns="p_type", values="n_total").reindex(columns=p_order)
pivot_cls  = meta_df.pivot(index="sigma", columns="p_type", values="n_classes").reindex(columns=p_order)

sns.heatmap(pivot_size, annot=True, fmt=",.0f", cmap="Blues", linewidths=0.4,
            cbar_kws={"label": "# images"}, ax=axes[0])
axes[0].set_xlabel("Population threshold (p-type)")
axes[0].set_ylabel("Sigma (σ)")

sns.heatmap(pivot_cls, annot=True, fmt="d", cmap="Purples", linewidths=0.4,
            cbar_kws={"label": "# ground-truth classes"}, vmin=0, vmax=5, ax=axes[1])
axes[1].set_xlabel("Population threshold (p-type)")
axes[1].set_ylabel("Sigma (σ)")

for ax in axes:
    ax.set_xticklabels(
        [f"{p}\n({PROBLEM_TYPE.get(p, '?')})" for p in p_order],
        rotation=0, fontsize=9,
    )

plt.tight_layout()
_save_fig(fig, "fig_pa01_dataset_overview")
plt.show()

In [ ]:
## 3  Sampling Parameters
# Strategy: keep ALL matched images in every fold evaluation (n_images = max = 50).
# Variance comes from resampling 60% of the per-image annotation pool each fold,
# then recomputing the modal sentiment from that subsample.

CV_FOLDS       = 5
SAMPLE_FRAC    = 0.6    # fraction of annotations per image per fold
BOOTSTRAP_SEED = 42

fold_rngs = [np.random.default_rng(BOOTSTRAP_SEED + i) for i in range(CV_FOLDS)]

print(f"CV_FOLDS={CV_FOLDS}, SAMPLE_FRAC={SAMPLE_FRAC}")
print("All matched images retained in every fold (n_images = max = 50)")
print("Per-fold variance: 60% annotation resample → modal recomputed each fold")

In [ ]:
## 4  Ground Truth Statistics
sigma_vals = sorted({int(k.split("_")[0].replace("sigma", "")) for k in datasets})

fig, axes = plt.subplots(len(sigma_vals), len(p_order), figsize=(14, 9), sharey=False)

for row_i, sigma in enumerate(sigma_vals):
    for col_j, p_type in enumerate(p_order):
        ax  = axes[row_i][col_j]
        key = f"sigma{sigma}_{p_type}"
        if key not in datasets:
            ax.set_visible(False)
            continue

        df_key = datasets[key]
        counts = df_key["ground_truth"].value_counts().sort_index()
        ax.bar([str(c) for c in counts.index], counts.values,
               color=PALETTE[0], alpha=0.75, edgecolor="white")
        ax.set_xlabel("ground_truth label", fontsize=7)
        ax.set_ylabel("images", fontsize=7)
        ax.tick_params(labelsize=7)

plt.tight_layout()
_save_fig(fig, "fig_pa02_gt_label_distribution")
plt.show()


In [ ]:
## 5  Load MLLM Annotations (raw, not pre-aggregated)
def load_jsonl(path: Path) -> pd.DataFrame:
    if not path.exists():
        print(f"WARNING: {path} not found.")
        return pd.DataFrame()
    records = [json.loads(line.strip()) for line in open(path, encoding="utf-8") if line.strip()]
    return pd.DataFrame(records)

ann_base = load_jsonl(OUTPUT_DIR / "annotations_baseline.jsonl")

# Keep only valid predictions; no aggregation yet — modal is recomputed per fold
ann_base = ann_base[ann_base["predicted_sentiment"].notna() &
                    (ann_base["predicted_sentiment"] != "")].copy()
ann_full = ann_full[ann_full["predicted_sentiment"].notna() &
                    (ann_full["predicted_sentiment"] != "")].copy()

def modal_per_image(ann: pd.DataFrame) -> pd.DataFrame:
    """Full-pool modal (used for confusion matrices)."""
    if ann.empty:
        return pd.DataFrame(columns=["image_id", "modal_sentiment"])
    return (ann.groupby("image_id")["predicted_sentiment"]
               .agg(lambda x: x.value_counts().index[0])
               .reset_index(name="modal_sentiment"))

def sampled_modal_per_image(ann: pd.DataFrame, frac: float, fold_rng) -> pd.DataFrame:
    """Resample frac of annotations per image, then compute modal."""
    if ann.empty:
        return pd.DataFrame(columns=["image_id", "modal_sentiment"])
    rows = []
    for img_id, grp in ann.groupby("image_id"):
        n = max(1, int(len(grp) * frac))
        sample = grp.sample(n=n, random_state=int(fold_rng.integers(0, 2**31)))
        modal  = sample["predicted_sentiment"].value_counts().index[0]
        rows.append({"image_id": img_id, "modal_sentiment": modal})
    return pd.DataFrame(rows)

modal_base_full = modal_per_image(ann_base)   # for confusion matrices
modal_full_full = modal_per_image(ann_full)

print(f"Baseline raw annotations : {len(ann_base):,}  across {ann_base['image_id'].nunique()} images")
print(f"Full-persona annotations : {len(ann_full):,}  across {ann_full['image_id'].nunique()} images")
print(f"Avg annotations/image (baseline): {len(ann_base)/ann_base['image_id'].nunique():.0f}")


In [ ]:
## 6  Evaluation — all images, 60% annotation resample per fold

def _compute_metrics(y_true, y_pred, problem, smap):
    m = {"accuracy": accuracy_score(y_true, y_pred)}
    # Use only labels present in y_true to avoid penalising absent classes
    # (e.g. Neutral disappears from P3 at σ=5, which would collapse macro-F1
    #  to 0.667 even when predictions on present classes are perfect).
    present_labels = sorted(set(y_true.tolist()))
    if problem == "binary":
        m["f1_macro"]    = f1_score(y_true, y_pred, average="macro",
                                    labels=present_labels, zero_division=0)
        m["f1_pos"]      = f1_score(y_true, y_pred, pos_label=1, zero_division=0)
        m["cohen_kappa"] = cohen_kappa_score(y_true, y_pred, weights=None)
        try:
            m["roc_auc"] = roc_auc_score(y_true, y_pred)
        except ValueError:
            pass
    elif problem == "multiclass-3":
        m["f1_macro"]    = f1_score(y_true, y_pred, average="macro",
                                    labels=present_labels, zero_division=0)
        m["cohen_kappa"] = cohen_kappa_score(y_true, y_pred, weights="linear")
    elif problem == "ordinal-5":
        m["f1_macro"]    = f1_score(y_true, y_pred, average="macro",
                                    labels=present_labels, zero_division=0)
        m["cohen_kappa"] = cohen_kappa_score(y_true, y_pred, weights="quadratic")
        m["mae"]         = mean_absolute_error(y_true, y_pred)
    return m


def evaluate_cv(key: str, p_type: str, df: pd.DataFrame,
                ann: pd.DataFrame, condition: str) -> dict:
    base = {"key": key, "condition": condition,
            "p_type": p_type, "problem": PROBLEM_TYPE.get(p_type, "?")}
    if ann.empty:
        return {**base, "note": "no annotations"}

    smap    = SENTIMENT_MAPS[p_type]
    problem = PROBLEM_TYPE[p_type]

    fold_results = []
    for fold_i, fold_rng in enumerate(fold_rngs):
        # Resample 60% of annotations per image → recompute modal
        modal_fold = sampled_modal_per_image(ann, SAMPLE_FRAC, fold_rng)

        # All images: GT ∩ resampled modal
        merged = df.merge(modal_fold, on="image_id", how="inner")
        if merged.empty or merged["ground_truth"].nunique() < 2:
            continue

        y_true = merged["ground_truth"].values
        y_pred = merged["modal_sentiment"].map(smap).values
        try:
            fold_results.append({
                **_compute_metrics(y_true, y_pred, problem, smap),
                "n_images": len(merged),
            })
        except Exception:
            continue

    if not fold_results:
        return {**base, "note": "insufficient data"}

    metrics_tracked = ["accuracy", "f1_macro", "cohen_kappa", "f1_pos", "roc_auc", "mae"]

    n_images_mean   = round(np.mean([r["n_images"] for r in fold_results]), 1)
    ann_per_img     = round(len(ann) / ann["image_id"].nunique() * SAMPLE_FRAC, 0)
    n_annotations   = int(n_images_mean * ann_per_img)   # total annotations per fold

    result = {
        **base,
        "n_images":      n_images_mean,
        "ann_per_image": ann_per_img,
        "n_annotations": n_annotations,
    }
    for metric in metrics_tracked:
        vals = [r[metric] for r in fold_results if metric in r]
        if vals:
            result[f"{metric}_mean"]    = round(np.mean(vals), 4)
            result[f"{metric}_std"]     = round(np.std(vals),  4)
            result[f"{metric}_ci_low"]  = round(np.percentile(vals, 2.5), 4)
            result[f"{metric}_ci_high"] = round(np.percentile(vals, 97.5), 4)

    return result


eval_rows = []
for key, df in datasets.items():
    p_type = key.split("_", 1)[1]
        eval_rows.append(evaluate_cv(key, p_type, df, ann, cond_label))

eval_df = pd.DataFrame(eval_rows)
eval_df["sigma"] = eval_df["key"].str.extract(r"sigma(\d+)").astype(int)

print(f"Evaluation complete — {len(eval_df)} rows\n")
print(eval_df[eval_df["condition"] == "baseline"][
    ["key", "n_images", "ann_per_image", "n_annotations", "f1_macro_mean", "cohen_kappa_mean"]
].to_string(index=False))

# ── Heatmaps ──────────────────────────────────────────────────────────────────
def _make_annot(sub, metric):
    val_piv = sub.pivot(index="sigma", columns="p_type", values=f"{metric}_mean").reindex(columns=p_order)
    std_piv = sub.pivot(index="sigma", columns="p_type", values=f"{metric}_std").reindex(columns=p_order)
    annot = val_piv.copy().astype(object)
    for idx in val_piv.index:
        for col in val_piv.columns:
            v, s = val_piv.loc[idx, col], std_piv.loc[idx, col]
            annot.loc[idx, col] = "—" if pd.isna(v) else f"{v:.3f}\n±{s:.3f}"
    return val_piv, annot

for cond, stem in [("baseline", "fig_pa03_metrics_baseline"),
    sub = eval_df[eval_df["condition"] == cond].copy()
    if sub.empty or "f1_macro_mean" not in sub.columns:
        continue
    fig, axes = plt.subplots(1, 2, figsize=(14, 3.8))
    for ax, (metric, label) in zip(axes, [("f1_macro", "Macro F1"), ("cohen_kappa", "Cohen's κ")]):
        if f"{metric}_mean" not in sub.columns:
            continue
        val_piv, annot = _make_annot(sub, metric)
        sns.heatmap(val_piv, annot=annot, fmt="", cmap="RdYlGn", vmin=0, vmax=1,
                    linewidths=0.4, cbar_kws={"label": label}, ax=ax)
        ax.set_ylabel("σ  (agreement threshold)")
        ax.set_xticklabels(
            [f"{p}\n({PROBLEM_TYPE.get(p,'?')})" for p in p_order],
            rotation=0, fontsize=8)
    plt.tight_layout()
    _save_fig(fig, stem)
    plt.show()


In [ ]:
## 7  Confusion Matrices — full annotation pool (all images)
valid_keys = sorted([
    k for k in datasets
    if k in eval_df[eval_df["condition"] == "baseline"]["key"].values
])

if valid_keys and not modal_base_full.empty:
    ncols = 4
    nrows = (len(valid_keys) + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 3.5, nrows * 3.5))
    axes = np.array(axes).flatten()

    for ax, key in zip(axes, valid_keys):
        p_type = key.split("_", 1)[1]
        smap   = SENTIMENT_MAPS[p_type]
        labels = sorted(set(smap.values()))
        if PROBLEM_TYPE[p_type] == "multiclass-3":
            labels = [1, 0, 2]

        merged = datasets[key].merge(modal_base_full, on="image_id", how="inner")
        if merged.empty:
            ax.set_visible(False)
            continue

        y_true = merged["ground_truth"].values
        y_pred = merged["modal_sentiment"].map(smap).values
        cm     = confusion_matrix(y_true, y_pred, labels=labels, normalize="true")

        tick_map = {
            "binary":       ["Neg", "Pos"],
            "multiclass-3": ["Neg", "Neu", "Pos"],
            "ordinal-5":    ["Neg", "SlNeg", "Neu", "SlPos", "Pos"],
        }
        sns.heatmap(cm * 100, annot=True, fmt=".1f", cmap="Blues",
                    xticklabels=tick_map[PROBLEM_TYPE[p_type]],
                    yticklabels=tick_map[PROBLEM_TYPE[p_type]],
                    linewidths=0.3, cbar=False, ax=ax)
        ax.set_xlabel("Predicted", fontsize=7)
        ax.set_ylabel("True", fontsize=7)
        ax.tick_params(labelsize=6)

    for ax in axes[len(valid_keys):]:
        ax.set_visible(False)

    plt.tight_layout()
    _save_fig(fig, "fig_pa05_confusion_baseline")
    plt.show()


In [ ]:
## 8  Summary Table
shared_cols = ["key", "p_type", "problem", "condition", "n_images", "ann_per_image"]
metric_cols = ["accuracy_mean", "f1_macro_mean", "cohen_kappa_mean",
               "f1_pos_mean", "roc_auc_mean", "mae_mean"]
avail_cols  = shared_cols + [c for c in metric_cols if c in eval_df.columns]

summary = (eval_df[avail_cols]
           .sort_values(["p_type", "key", "condition"])
           .reset_index(drop=True))

fmt_dict = {c: "{:.4f}" for c in metric_cols if c in summary.columns}
display(
    summary.style
    .set_caption(f"Agreement metrics — all images, 60% annotation resample, mean over {CV_FOLDS} folds")
    .hide(axis="index")
    .format(fmt_dict, na_rep="—")
)

# ── Export LaTeX ──────────────────────────────────────────────────────────────
tex = summary.to_latex(
    index=False,
    caption=(
        "Evaluation metrics per agreement subset $(\\sigma, p)$ and condition. "
        f"All matched images retained; variance from {CV_FOLDS} folds of 60\\% "
        "annotation resampling per image (\\texttt{random\\_seed}=42). "
        "Cohen's $\\kappa$ weighting: unweighted (binary), linear (3-class), "
        "quadratic (ordinal-5). "
        "Macro F1 is the primary metric following MLLMsent (arXiv:2508.16873)."
    ),
    label="tab:perceptsent_agreement_eval",
    escape=True,
    float_format="{:.4f}".format,
    na_rep="---",
)
tex_path = FIGURES_DIR / "table3_perceptsent_agreement_eval.tex"
tex_path.write_text(tex)
print(f"LaTeX table saved → {tex_path}")
